# LangGraph + TypeSafe

**Difficulty:** Intermediate

LangGraph orchestrates a workflow as a graph of nodes and edges; TypeSafe makes narrow, structured decisions. Neither one needs the other, but they compose well at one specific seam: LangGraph's *conditional edges* have to turn "what does this state mean" into the name of the next node to run, and a TypeSafe `Choice` — a closed-set answer with a `confidence` score — is a better fit for that than parsing free-text LLM output by hand.

This cookbook is original to this repo, not a `docs.typesafe.ai` cookbook — there isn't one. It's grounded in LangGraph's own docs (see `research/langgraph.md`) plus this project's own confidence-gated routing and intent-routing lessons.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.langchain.com/oss/javascript/langgraph — overview, graph-api, thinking-in-langgraph (click to expand)</summary>

# LangGraph

> LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents. LangGraph gives you fine-grained control to mix deterministic, hand-coded steps with LLM-driven agentic steps in the same graph.

Install: `npm install @langchain/langgraph @langchain/core`.

## Graphs

At its core, LangGraph models agent workflows as graphs, defined with three components:

1. **State**: a shared data structure representing the current snapshot of the application, typically defined with a shared state schema.
2. **Nodes**: functions that encode the logic of your agents. They receive the current state as input, perform some computation, and return an updated state.
3. **Edges**: functions that determine which node to execute next based on the current state. They can be conditional branches or fixed transitions.

*Nodes do the work, edges tell what to do next.* `Nodes` and `Edges` are nothing more than functions — they can contain an LLM call or just ordinary code.

### StateGraph and compiling

`StateGraph` is the main graph class, parameterized by a `StateSchema`. You define state, add nodes and edges, then compile: "You **MUST** compile your graph before you can use it."

```typescript
const graph = new StateGraph(State)
  .addNode("nodeA", nodeA)
  .addEdge(START, "nodeA")
  .addEdge("nodeA", END)
  .compile();
```

### State and reducers

Each field in a `StateSchema` has a reducer function specifying how a node's update is applied to it. By default, "the default reducer ignores the left argument and replaces the state value with the right argument" — an update simply overwrites. A custom reducer can accumulate instead (e.g. concatenating arrays), for cases where parallel nodes need to merge their updates rather than one clobbering another.

### Edges

- **Normal edges** (`addEdge("nodeA", "nodeB")`): always go from A to B.
- **Conditional edges** (`addConditionalEdges("nodeA", routingFunction)`): call a function to determine which node(s) to go to next. By default, the routing function's return value is used as the name of the next node; you can also supply a third-argument mapping from the function's return value to a node name.
- **Entry point** (`addEdge(START, "nodeA")`) and **conditional entry point** (`addConditionalEdges(START, routingFunction)`): which node (or which node, decided dynamically) runs first.

> For each node, choose one routing mechanism: use normal edges for static routing, or use conditional edges for dynamic routing. Do not mix normal edges and dynamic routing from the same node, because both paths can execute and make graph behavior harder to reason about.

The docs describe conditional edges mechanically — a function of `state` that returns a node name — but don't prescribe how that function should *decide*. LangGraph's applied design guide, `thinking-in-langgraph`, does show an LLM feeding that decision (a `classifyIntent` node routes on a structured LLM classification) — but its own escalation rule is a hardcoded intent/urgency check, never a confidence threshold. That's the seam this cookbook actually fills.

That same guide also states a design principle this cookbook follows: "keep state raw, format prompts on-demand" — state should hold plain data, not pre-formatted prompt text — and recommends its own routing live *inside* a node via `Command({ update, goto })` rather than in a separate function passed to `addConditionalEdges`, "because routing happens inside nodes... making the flow explicit and traceable."

> The full guides also cover `Send` (fan-out to a variable number of node invocations), persistence via checkpointers, streaming, and human-in-the-loop interrupts — trimmed here to the vocabulary this cookbook actually uses. Read them at [docs.langchain.com/oss/javascript/langgraph/overview](https://docs.langchain.com/oss/javascript/langgraph/overview), [.../graph-api](https://docs.langchain.com/oss/javascript/langgraph/graph-api), and [.../thinking-in-langgraph](https://docs.langchain.com/oss/javascript/langgraph/thinking-in-langgraph) any time.

</details>

## The core idea

A conditional edge's routing function has to return the name of the next node — a value from a fixed, closed set. That's exactly the shape a TypeSafe `Choice` question already returns: pick one option from a defined set, plus a `confidence` for how sure the model was. Instead of asking an LLM a free-text question and parsing its reply to decide which branch to take, the routing function can read a `Choice` answer's `choice` field directly.

LangGraph's own applied design guide (`docs.langchain.com/oss/javascript/langgraph/thinking-in-langgraph`, see `research/langgraph.md`) does show an LLM feeding a routing decision — its worked support-email example classifies intent with a structured LLM call and routes on the result. What it doesn't do is confidence-gate that routing: its escalation rule is a hardcoded `intent === "billing" || urgency === "critical"` check, which means the model has to separately decide *and report* an `urgency` field just so the application knows when to trust it less. A TypeSafe `Choice`'s `confidence` gives you that "how sure was I" signal on the *same* answer, for free — no second field to design, elicit, and validate. That's the actual, narrower gap this cookbook fills: not "LLM-fed routing doesn't exist in the docs," but "confidence-gated routing doesn't."

That guide also confirms two things this cookbook already does, without having read it first: state stores raw fields (`category`, `confidence`), never formatted prompt text — its own stated principle, "keep state raw, format prompts on-demand" — and `classify` is kept as its own node rather than folded into a larger one, for the same reason it gives (isolating the LLM call for its own visibility and debugging, independent of what happens downstream).

One thing it does differently: its own routing lives *inside* each node, returned as `Command({ update, goto })`, rather than in a separate function passed to `addConditionalEdges` the way this cookbook does. Both are documented, valid LangGraph patterns. This cookbook keeps them separate on purpose — to keep "what TypeSafe decided" and "what the graph does about it" visibly distinct, matching this repo's recurring code-decides/model-decides split — but it's a real design fork, not a settled-by-the-docs answer. See `research/langgraph.md` if you want to try rewriting `classify` as a `Command`-returning node instead.

Nothing about this needs LangGraph's more advanced machinery beyond that one fork — no custom reducers, no persistence, no `Send`. One `StateGraph`, one node that calls TypeSafe, one conditional edge whose routing function reads that node's output.

In [1]:
import { client } from "../../assets/typesafe-client.ts";
import { choice } from "npm:@typesafe-ai/sdk";
import { StateGraph, StateSchema, START, END } from "npm:@langchain/langgraph";
import type { GraphNode } from "npm:@langchain/langgraph";
import * as z from "npm:zod";

const CONFIDENCE_FLOOR = 0.6;

const State = new StateSchema({
  ticket: z.string(),
  category: z.string().optional(),
  confidence: z.number().optional(),
  resolution: z.string().optional(),
});

// The seam: one TypeSafe Choice call, its answer written straight into graph state.
const classify: GraphNode<typeof State> = async (state) => {
  const response = await client.systemOne({
    state: state.ticket,
    questions: {
      category: choice("Determine the broad category of this support ticket", {
        billing: "Charges, invoices, refunds, subscriptions",
        account: "Login, permissions, profile, security",
        feature_request: "The user is requesting new functionality",
      }),
    },
  });
  const { category } = response.answers;
  return { category: category.choice, confidence: category.confidence };
};

// The routing function: reads the Choice answer's confidence before trusting its category.
function route(state: typeof State.State): string {
  if ((state.confidence ?? 0) < CONFIDENCE_FLOOR) return "humanReview";
  return state.category ?? "humanReview";
}

const handleBilling: GraphNode<typeof State> = () => ({ resolution: "routed to billing team" });
const handleAccount: GraphNode<typeof State> = () => ({ resolution: "routed to account team" });
const handleFeature: GraphNode<typeof State> = () => ({ resolution: "logged as feature request" });
const humanReview: GraphNode<typeof State> = () => ({ resolution: "flagged for human review" });

const graph = new StateGraph(State)
  .addNode("classify", classify)
  .addNode("billing", handleBilling)
  .addNode("account", handleAccount)
  .addNode("feature_request", handleFeature)
  .addNode("humanReview", humanReview)
  .addEdge(START, "classify")
  .addConditionalEdges("classify", route, {
    billing: "billing",
    account: "account",
    feature_request: "feature_request",
    humanReview: "humanReview",
  })
  .addEdge("billing", END)
  .addEdge("account", END)
  .addEdge("feature_request", END)
  .addEdge("humanReview", END)
  .compile();

const tickets: Record<string, string> = {
  clearAccountIssue: "I can't log into my account since the password reset yesterday.",
  genuinelyAmbiguous: "Hey, something's off with my order and I'm not sure who to ask.",
};

for (const [name, ticket] of Object.entries(tickets)) {
  const result = await graph.invoke({ ticket });
  console.log(name, result);
}

clearAccountIssue {
  ticket: "I can't log into my account since the password reset yesterday.",
  category: "account",
  confidence: 1,
  resolution: "routed to account team"
}
genuinelyAmbiguous {
  ticket: "Hey, something's off with my order and I'm not sure who to ask.",
  category: "billing",
  confidence: 0.88,
  resolution: "routed to billing team"
}


## What just happened

Two tickets through the same graph. `clearAccountIssue` names a specific, ordinary account action (a password reset): `classify` landed on `account` at confidence 1.0, and the conditional edge routed straight to the `account` leaf node — no surprises there.

`genuinelyAmbiguous` is the interesting one, and it didn't go the way its name predicts. It was written to withhold any obvious category signal — but it resolved to `billing` at confidence 0.88, comfortably clear of `CONFIDENCE_FLOOR` (0.6), and routed straight to `billing` instead of `humanReview`. The word "order" alone was apparently enough of a billing signal to the model, with no charge or refund mentioned. That's the actual lesson here: a ticket that reads as ambiguous *to a person* doesn't guarantee low `confidence` — confidence measures how concentrated the model's own probability distribution is over its options, not how hedgy the prose sounds to a human reader. Don't design a confidence floor around your own intuition of what's ambiguous; design it around what your actual traffic's confidence distribution looks like.

The graph structure never changes between the two runs — the same five nodes and one conditional edge handle both. Only the state flowing through it differs. That's the LangGraph half of the point as much as the TypeSafe half: the routing *decision* is data (a `Choice` answer written into state), not a hardcoded branch, so the graph doesn't need a special case for "ambiguous ticket" — it already has one, because `confidence` is just another field on state like any other.

**Try it yourself:** rewrite `genuinelyAmbiguous` to actually land below `0.6` — try removing "order" entirely, or adding a second ticket that genuinely straddles two categories (e.g. mentions both a login problem and a refund) — and confirm it's the one that reaches `humanReview`. Then add a fourth node and a fourth category (e.g. `spam`) and confirm you only need to touch the `classify` node's `Choice` criteria and the conditional edge's mapping — nothing else in the graph's shape changes.

## Related

- `lessons/patterns/0007-confidence-gated-routing.ipynb` — the confidence-floor idea this cookbook's `CONFIDENCE_FLOOR` applies inside a LangGraph conditional edge instead of a plain `if`
- `lessons/patterns/0009-intent-routing.ipynb` — the same classify-then-dispatch shape, without a graph framework wrapped around it
- `lessons/patterns/0006-speculative-fan-out.ipynb` — the ticket-triage example this cookbook's `classify` node is deliberately close to
- `research/langgraph.md` — the primary-source research this cookbook was built from, including the `Command`-vs-conditional-edges design fork and what LangGraph's own applied guide does and doesn't say about confidence-gated routing

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/langgraph.md` and `research/typesafe-ai.md`.